# Lista 3 - MVVM, ViewModel i strumienie danych

**Termin wykonania:** gr. wt. 01.12.2026; gr. czw. 03.12.2026

**Zadanie praktyczne: 6 pkt. Odpowiedź ustna: 4 pkt. Łącznie: 10 pkt.**

## Zadanie 1 - Monitor pobierania (6 pkt)

Zaimplementuj w Kotlinie i Jetpack Compose aplikację symulującą pobieranie plików. Lista plików i ich rozmiary są przykładowe. Postęp wyznacza lokalna symulacja oparta na `delay()`, bez internetu i rzeczywistego zapisu plików.

Zastosuj MVVM, repozytorium przekazywane przez konstruktor ViewModelu, zimny `Flow` jako źródło postępu, `StateFlow` jako stan aplikacji oraz `SharedFlow` do ulotnych powiadomień.

### 1. Dane, repozytorium i podział odpowiedzialności - 1 pkt

- Przygotuj dane dla co najmniej 4 symulowane pliki z unikatowymi identyfikatorami, nazwami i rozmiarami w podanej jednostce, np. MB. Zdefiniuj własne modele danych (dummy data).
- Repozytorium udostępnia dane plików i symulację pobierania. Nie powinno zależeć od `Composable`, stanu kontrolek ani `Activity`.
- ViewModel obsługuje akcje użytkownika i udostępnia stan. Funkcje wyświetlające ekran otrzymują dane oraz callbacki; nie uruchamiają samodzielnie pobierania z repozytorium.
- Przekaż repozytorium do konstruktora ViewModelu. Instancję ViewModelu pobieraj przez `viewModel()` z inicjalizatorem lub fabryką.

### 2. Zimny Flow z postępem pobierania - 1 pkt

- Zaimplementuj w repozytorium funkcję `download(fileId: Int): Flow<Int>`. Zwracany strumień utwórz przez `flow { ... }`.
- Po rozpoczęciu zbierania wyemituj `0`, a następnie `25`, `50`, `75` i `100`. Przed każdą z czterech kolejnych emisji wykonaj `delay(1000)`. Po emisji `100` strumień ma się poprawnie zakończyć.
- Samo wywołanie `download(fileId)` nie rozpoczyna symulacji. Każde nowe zbieranie strumienia rozpoczyna własny przebieg od `0`; repozytorium nie może używać jednego globalnego licznika postępu dla wszystkich przebiegów.
- Zbieranie wykonuje ViewModel. Nie używaj `Thread.sleep()`, `runBlocking` w kodzie UI ani `GlobalScope`. Interfejs musi reagować przez cały czas.

### 3. Stan aplikacji przez StateFlow - 1 pkt

- Przygotuj stan ekranu zawierający pliki oraz postęp i status każdego pliku. Rozróżnij przynajmniej: „Oczekuje”, „Pobieranie”, „Zakończono” i „Anulowano”. Początkowo każdy plik oczekuje i ma postęp `0%`.
- ViewModel przechowuje prywatny `MutableStateFlow` i udostępnia publiczny `StateFlow` tylko do odczytu, np. przez `asStateFlow()`. UI nie może samodzielnie zmieniać `.value` źródła.
- Odbieraj stan w Compose przez `collectAsStateWithLifecycle()`. Wyświetl nazwę, rozmiar, aktualny procent, wskaźnik postępu i status każdego pliku.
- Kolejne wartości zimnego strumienia aktualizują stan właściwego pliku. Twórz nowe wartości stanu i list, np. przez `copy` i `map`, zamiast modyfikować istniejącą listę w miejscu. Zakończenie pobierania ma być widoczne w stanie również wtedy, gdy nikt nie odbiera powiadomień.

### 4. Sterowanie pobieraniem i czas życia ViewModelu - 1 pkt

- Dodaj przyciski „Pobierz” lub „Pobierz ponownie” oraz „Anuluj”. W całej aplikacji może trwać najwyżej jedno pobieranie naraz. Pozostałe przyciski rozpoczęcia są wtedy nieaktywne; ViewModel również ma chronić przed uruchomieniem drugiego zadania.
- Pobieranie uruchamiaj w `viewModelScope`, zachowując jego `Job`. Anulowanie zatrzymuje zbieranie postępu, ustawia status „Anulowano” i pozostawia ostatni odebrany procent. Pobierania już zakończonego nie oznaczaj jako anulowanego.
- Ponowne pobranie wybranego pliku zaczyna się od `0%`. Wyniki i statusy pozostałych plików pozostają bez zmian.
- Rekompozycja i obrót urządzenia nie mogą uruchamiać dodatkowego pobierania ani resetować stanu ViewModelu. Trwająca operacja ma być kontynuowana przy odtworzeniu Activity. Samo przejście aplikacji do tła nie kończy zadania w ViewModelu. Trwały zapis po zakończeniu procesu aplikacji nie jest wymagany.

### 5. Ulotne powiadomienia przez SharedFlow - 1 pkt

- Zdefiniuj własny typ zdarzenia zawierający co najmniej identyfikator zdarzenia i treść komunikatu. Identyfikatory mają być unikatowe w czasie życia danej instancji ViewModelu.
- W ViewModelu utwórz prywatny `MutableSharedFlow` z `replay = 0` i `extraBufferCapacity = 0`. Udostępnij go UI tylko do odczytu przez `asSharedFlow()`. Emituj zdarzenia przez `emit()` z korutyny.
- Wyślij komunikaty o rozpoczęciu, zakończeniu i anulowaniu pobierania. Dodaj też przycisk „Test powiadomienia”, działający niezależnie od pobierania. Kolejne naciśnięcia tego przycisku mają emitować tę samą treść z kolejnymi identyfikatorami.
- Powiadomienia są ulotne: przy braku odbiorców nie czekają na przyszłą subskrypcję. Status i postęp pobierania pozostają źródłem informacji o wyniku operacji; nie zastępuj ich samym zdarzeniem ani historią komunikatów.

### 6. Dwóch niezależnych odbiorców powiadomień - 1 pkt

- Dodaj na ekranie dwa panele: „Odbiorca A” i „Odbiorca B”. Oba korzystają z tego samego strumienia zdarzeń i tej samej instancji ViewModelu. Każdy ma własny przycisk „Dołącz” / „Opuść”, licznik odebranych zdarzeń oraz ostatni odebrany identyfikator i komunikat.
- Każdy dołączony panel ma własny kolektor `collect`. Odłączenie jednego panelu nie zatrzymuje drugiego ani pobierania. Licznik odłączonego panelu nie rośnie. Ponowne dołączenie nie odtwarza pominiętych komunikatów, ponieważ `replay = 0`.
- Zbieraj zdarzenia tylko wtedy, gdy panel jest dołączony, a właściciel ekranu ma co najmniej stan `STARTED`. Możesz połączyć `LaunchedEffect` z `repeatOnLifecycle(Lifecycle.State.STARTED)`. Nie zastępuj dwóch kolektorów jednokrotnym odczytem `.value` ani kopiowaniem komunikatu z panelu A do B.
- Pokaż faktyczną liczbę aktywnych odbiorców strumienia zdarzeń, korzystając z `subscriptionCount`. W trakcie prezentacji po dołączeniu panelu zaczekaj na zmianę tego licznika, zanim wyślesz kolejne zdarzenie. Samo ustawienie zmiennej UI nie potwierdza jeszcze rozpoczęcia subskrypcji.
- Stan dołączenia paneli zachowaj przy obrocie, np. przez `rememberSaveable`. Lokalne liczniki i ostatnie komunikaty mogą się wtedy wyzerować. Stan plików i trwające pobieranie mają zachować się zgodnie z częścią 4.

Wskazówka: `collectAsStateWithLifecycle()` służy tu do odbioru stanu, a osobne kolektory do wykonania reakcji na każde zdarzenie. Opis `repeatOnLifecycle` znajduje się w [dokumentacji zbierania Flow w cyklu życia UI](https://developer.android.com/kotlin/flow/stateflow-and-sharedflow).

### Punktacja zadania i prezentacja rozwiązania

Każda z sześciu części zadania jest oceniana osobno, maksymalnie na **1 pkt**. Liczba losowanych pytań nie zależy od liczby punktów zdobytych za aplikację. Do kodu mogą pojawić się dodatkowe pytania sprawdzające zrozumienie i samodzielne wykonanie implementacji.

Na laboratorium uruchom aplikację na emulatorze lub urządzeniu i przygotuj kod do omówienia. Pokaż następujące scenariusze:

| Scenariusz | Oczekiwany wynik |
|:---|:---|
| Pobierz wybrany plik do końca. | Postęp przechodzi przez `0`, `25`, `50`, `75`, `100`; końcowy status to „Zakończono”. |
| Podczas pobierania spróbuj rozpocząć inne. | W aplikacji nadal działa najwyżej jedna operacja pobierania. |
| Anuluj pobieranie przed `100%`, a potem uruchom je ponownie. | Postęp po anulowaniu nie rośnie; kolejna próba zaczyna od `0%`. |
| Obróć urządzenie w trakcie pobierania. | Postęp nie resetuje się, operacja jest kontynuowana i nie powstaje drugie zadanie. |
| Przejdź do tła na czas dłuższy niż pozostały czas pobierania, a potem wróć. | Po powrocie UI otrzymuje aktualny stan „Zakończono”. |
| Odłącz obu odbiorców, wyślij testowy komunikat i dołącz A. | A nie otrzymuje starego komunikatu. Sam brak odbiorców nie blokuje emitera. |
| Przy dołączonym A wyślij testowy komunikat, potem dołącz B. | A odbiera komunikat; B nie otrzymuje go po dołączeniu. |
| Przy obu dołączonych panelach dwukrotnie naciśnij „Test powiadomienia”. | Oba panele odbierają dwa zdarzenia o różnych identyfikatorach, mimo identycznej treści. |
| Odłącz B i wyślij kolejne zdarzenie. | Rośnie tylko licznik A. Po ponownym dołączeniu B nie nadrabia zaległego zdarzenia. |

## Odpowiedź ustna - 4 pkt

- Student losuje **4 różne pytania** z całej puli **1-26**. Podział tematyczny służy uporządkowaniu materiału i nie zmienia zasad losowania.
- Za każde pytanie można uzyskać **0 albo 1 pkt**. Nie przyznaje się punktów częściowych za odpowiedź.
- Losowanie obejmuje zawsze 4 pytania, niezależnie od wyniku zadania praktycznego. Maksymalny wynik odpowiedzi ustnej to 4 pkt.
- Odpowiedź powinna wyjaśniać działanie mechanizmu. Wymagany przykład, przewidywany wynik lub uzasadnienie stanowi część odpowiedzi.

## Pula pytań do losowania

Każde pytanie jest warte **1 pkt**. Obowiązuje cała pula **1-26**. Zakres: wykłady 4 i 5 PUM2 oraz podstawy z PUM1 i wcześniejszych wykładów PUM2 potrzebne do zrozumienia omawianych mechanizmów.

### A. MVVM i repozytorium

1. Jakie odpowiedzialności mają View, ViewModel i Model w aplikacji opartej na MVVM? **(1 pkt)**
2. Na czym polega jednokierunkowy przepływ danych między UI i ViewModelem? Opisz drogę od kliknięcia przycisku do wyświetlenia nowego stanu. **(1 pkt)**
3. Jaką rolę pełni repozytorium i dlaczego ViewModel nie powinien sam wybierać sposobu dostępu do każdego źródła danych? **(1 pkt)**
4. Co daje przekazanie repozytorium do ViewModelu przez konstruktor zamiast tworzenia go bezpośrednio w metodzie obsługującej kliknięcie? **(1 pkt)**
5. Dlaczego funkcji wyświetlającej ekran warto przekazać dane i callbacki zamiast pozwalać jej bezpośrednio wywoływać repozytorium? **(1 pkt)**
6. Dlaczego `private set` na właściwości typu `MutableStateFlow` nie blokuje UI możliwości zmiany jej `.value`? **(1 pkt)**
7. Dlaczego ViewModel nie powinien przechowywać referencji do Activity ani kontrolek interfejsu? **(1 pkt)**

### B. ViewModel i czas życia stanu

8. Co AndroidX ViewModel daje ponad zwykłą klasę przechowywaną przez `remember`? **(1 pkt)**
9.  Od czego zależy, czy dwa wywołania `viewModel()` zwrócą tę samą instancję? Uwzględnij właściciela i klucz. **(1 pkt)**
10. Co dzieje się ze stanem i korutyną w `viewModelScope` przy odtworzeniu Activity wskutek obrotu urządzenia? **(1 pkt)**
11. Kiedy zadania w `viewModelScope` są automatycznie anulowane? Czy sam brak widoczności ekranu zawsze oznacza usunięcie ViewModelu? **(1 pkt)**
12. Dlaczego zatrzymanie zbierania przez UI nie musi zatrzymać producenta uruchomionego w `viewModelScope`? **(1 pkt)**

### C. Flow, StateFlow i odbiór stanu

12. Kiedy zaczyna wykonywać się blok strumienia utworzonego przez `flow { ... }`: podczas utworzenia obiektu czy rozpoczęcia zbierania? **(1 pkt)**
18. Dwie korutyny zbierają ten sam zimny strumień `flow { ... }`. Czy współdzielą jedno wykonanie jego bloku? Uzasadnij. **(1 pkt)**
19. Czy funkcja `download(fileId: Int): Flow<Int>` musi mieć modyfikator `suspend`, jeśli zwraca strumień utworzony przez `flow { delay(...); emit(...) }`? Wyjaśnij. **(1 pkt)**
20. Do czego służy `emit()` w bloku `flow` i co reprezentują kolejne emisje w przykładzie postępu pobierania? **(1 pkt)**
21. Dlaczego zbierania przez `collect` nie rozpoczynamy zwykłym wywołaniem w ciele funkcji Composable? Gdzie można je uruchomić? **(1 pkt)**
22. Dlaczego `MutableStateFlow` wymaga wartości początkowej i do czego służy właściwość `.value`? **(1 pkt)**
23. Pobieranie osiągnęło `75%`, a nowy kolektor zaczyna obserwować jego `StateFlow`. Jaką wartość otrzyma na początku? **(1 pkt)**
24. Co zrobi `StateFlow`, gdy przypiszemy wartość równą poprzedniej według `equals`? Dlaczego jest to istotne przy aktualizowaniu list? **(1 pkt)**
25. Jaką rolę pełni `collectAsStateWithLifecycle()` i co dzieje się ze zbieraniem po zejściu poniżej domyślnego progu `STARTED`? **(1 pkt)**

### D. SharedFlow i powiadomienia

21. Dwa aktywne kolektory obserwują ten sam `SharedFlow`. Czy nowe zdarzenie jest przekazywane obu, czy tylko jednemu z nich? **(1 pkt)**
28. Wywołano `emit()` na `MutableSharedFlow(replay = 0)` bez odbiorców i bez dodatkowego bufora. Czy zdarzenie zaczeka na przyszłego subskrybenta? **(1 pkt)**
29. Czym różni się dołączenie nowego odbiorcy do `SharedFlow` z `replay = 0` od dołączenia przy `replay = 1`? **(1 pkt)**
30. Czy `SharedFlow` automatycznie pomija dwie kolejne równe wartości tak jak `StateFlow`? Co pokażą liczniki odbioru po dwóch identycznych komunikatach? **(1 pkt)**
31. Czy `extraBufferCapacity` przechowuje historię dla przyszłych subskrybentów? Do jakiej sytuacji służy ten parametr? **(1 pkt)**
32. Jaką decyzję określa `onBufferOverflow`? Wyjaśnij różnicę między zawieszeniem emitera a odrzuceniem wartości. **(1 pkt)**

## Oceny

Punkty za zadanie i odpowiedzi sumują się.

| Punkty łącznie | Ocena |
|:---|:---:|
| poniżej 6 | 2,0 |
| 6 | 3,0 |
| 7 | 3,5 |
| 8 | 4,0 |
| 9 | 4,5 |
| 10 | 5,0 |